In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Загружаем данные

In [ ]:
# Загружаем только необходимые колонки
df = pd.read_excel('data_orphans_127_v20241218.xlsx', 
        usecols=['year', 'region', 'indicator_category', 'dimension', 
        'main_level', 'indicator', 'indicator_value'])

# Быстро проверим размер
print(df.shape)
print(df.head(2))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
"""Горизонт - год, вертикаль - число сирот"""
plt.plot(specs_by_year['year'], specs_by_year['indicator_value'], marker='o', linestyle='-')
plt.title('Динамика численности специалистов госсектора, занимающихся решением проблемы')
plt.xlabel('Год')
plt.ylabel('Специалисты')
plt.grid(True)
plt.show()

# Выявить все строки таблицы исходя из любых значений аттрибутов

хм, окей, однако вернемся теперь к предыдущей задаче
предварительно мы должны сделать такой модуль-программу, который нам позволит видеть (лучше даже автоматически при запуске самому проверять), какие мы действительно можем строки выводит, а каких комбинаций у нас не существует
И желательно выводить с примерами
Давай сначала разберемся на элементарном нашем целевом примере:
- Отмены решений о передаче в семью (возврат детей, ранее устроенных в семьи, в органы опеки)
> Решение:
attr_list = ['year', 'indicator_category', 'dimension', 'main_level', 'indicator', 'indicator_value']

# Для конструкта нашего target_df
target_mask = [0, 1, 1, 1, 0, 0]
#Предполагаемые значения:
target_vals = ['3. Устройство детей в семьи', 'устроено в семьи, всего', 'отменено решений о передаче ребенка на воспитание в семью, всего']
# Т.е. предполагаемый target_df, который автоматически крафтится:
target_df = [df['indicator_category'] == '3. Устройство детей в семьи' & df ['dimension'] == 'устроено в семьи, всего' & df['main_level'] == 'отменено решений о передаче ребенка на воспитание в семью, всего']
]
___
Однако наш df может оказаться некорректным или не оказаться. Для этого нам нужно составить предварительный блок-алгоритм:
> Составить сводную таблицу если существуют строки со значениями indicator такие, что наши целевые (indicator_category, demension, main_level) имею собственно строку
1) Составляем ind_cat -> demension
2) ind_cat -> main_level
3) demension (тип связи?) --- main_level
 > в зависимости от типа связи (скорее всего подразумевается demension : main_level = 1 : many)
мы составляем парную строку и определяем в какой cat_ind эта пара...

In [ ]:
attr_list = ['year', 'indicator_category', 'dimension', 'main_level', 'indicator', 'indicator_value']
df_target = [inp]

>FIXME:

# Автоматизация обработки данных
## Шаг 1. Поиск показателя «Отмены решений о передаче в семью»

Выполните поиск по ключевому слову `"отмен"` в колонке `main_level`:

```python
mask_otmen = df['main_level'].str.contains('отмен', na=False, case=False)
df_otmen = df[mask_otmen]
print(df_otmen[['indicator_category', 'dimension', 'main_level', 'indicator']].drop_duplicates().to_string(max_colwidth=60))
```

**Ожидаемый результат:**  
Вы увидите несколько строк, например:
- `'отменено решений о передаче ребенка на воспитание в семью, всего'` (скорее всего, это то, что нужно)
- `'отменено решений о передаче ребенка на воспитание в семью по инициативе органа опеки и попечительства, всего'`
- `'отменено решений о передаче ребенка на воспитание в семью, по инициативе усыновителей...'`

В ТЗ сказано: **«Отмены решений о передаче в семью (возврат детей)»** — это общее количество отмен, независимо от инициатора. Значит, нужно брать строку, где `main_level = 'отменено решений о передаче ребенка на воспитание в семью, всего'`.

## Шаг 2. Проверка комбинации (indicator_category, dimension, main_level)

Посмотрите, в какой категории находится этот показатель. Скорее всего, `indicator_category` будет `'3. Устройство детей в семьи'`, а `dimension` — `'устроено в семьи, всего'` или `'всего'`. Уточните, выполнив:

```python
# Фильтр по точному main_level
mask_exact = df['main_level'] == 'отменено решений о передаче ребенка на воспитание в семью, всего'
df_exact = df[mask_exact]
print(df_exact[['indicator_category', 'dimension', 'year', 'indicator_value']].head())
```

Вы увидите, какие значения `indicator_category` и `dimension` встречаются. Запомните их.

## Шаг 3. Построение динамики отмен по России

Используя найденные точные названия, напишите фильтр:

```python
mask = (df['indicator_category'] == '3. Устройство детей в семьи') & \
       (df['dimension'] == 'устроено в семьи, всего') & \
       (df['main_level'] == 'отменено решений о передаче ребенка на воспитание в семью, всего')

df_cancellations = df[mask]
canc_by_year = df_cancellations.groupby('year')['indicator_value'].sum().reset_index()

plt.figure(figsize=(10,5))
plt.plot(canc_by_year['year'], canc_by_year['indicator_value'], marker='o', color='red')
plt.title('Отмены решений о передаче детей в семьи (возвраты)')
plt.xlabel('Год')
plt.ylabel('Количество отмен')
plt.grid(True)
plt.show()
```

Если данных нет (пустой DataFrame), возможно, `dimension` не `'устроено в семьи, всего'`, а `'всего'`. Проверьте по выводу из шага 2.

## Шаг 4. Автоматическая проверка существования комбинации (ваш блок-алгоритм)

Вы хотите, чтобы программа сама проверяла, существует ли комбинация. Вот функция:

```python
def combo_exists(df, conditions):
    """
    conditions: список кортежей (column, value)
    возвращает: (True/False, dataframe отфильтрованный)
    """
    mask = pd.Series([True] * len(df))
    for col, val in conditions:
        mask &= (df[col] == val)
    result = df[mask]
    return (len(result) > 0, result)

# Пример использования
conditions = [
    ('indicator_category', '3. Устройство детей в семьи'),
    ('dimension', 'устроено в семьи, всего'),
    ('main_level', 'отменено решений о передаче ребенка на воспитание в семью, всего')
]
exists, data = combo_exists(df, conditions)
print(f"Существует: {exists}")
if exists:
    print(data[['year', 'region', 'indicator_value']].head())
```

Если `exists` ложно, вы можете попробовать другие варианты `dimension` (например, `'всего'`).

## Шаг 5. Сводная таблица связей (для навигации)

Вы предложили построить таблицу, какие `dimension` и `main_level` встречаются вместе в каждой категории. Это можно сделать так:

```python
# Для каждой категории собираем уникальные пары (dimension, main_level)
pairs = df.groupby('indicator_category').apply(
    lambda g: g[['dimension', 'main_level']].drop_duplicates().to_dict('records')
)
for cat, recs in pairs.items():
    print(f"\n=== {cat} ===")
    for r in recs[:5]:  # покажем первые 5
        print(f"  dimension: {r['dimension']} -> main_level: {r['main_level']}")
    if len(recs) > 5:
        print(f"  ... и ещё {len(recs)-5} пар")
```

Это поможет вам в будущем быстро находить нужные комбинации без поиска по ключевым словам.

## Ваше действие

1. Выполните **Шаг 1** и **Шаг 2**, чтобы точно определить значения для показателя "Отмены".
2. Если они отличаются от моих предположений, скорректируйте фильтр в **Шаге 3**.
3. Постройте график и убедитесь, что данные есть.
4. Сохраните этот фильтр для дальнейшего использования.

Если на каком-то этапе получите пустой результат, не стесняйтесь спрашивать. Мы разберёмся.